In [46]:
import pandas as pd
import numpy as np
from pathlib import Path

In [47]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [48]:
PROJECT_ROOT=Path("../../").resolve()

DATA_DIR =PROJECT_ROOT/"data"/"processed"
OUTPUT_DIR = PROJECT_ROOT/"ml"/"outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [49]:
required_files = [
    "invoices.csv",
    "shipments_evidence.csv",
    "load_evidence.csv",
    "delivery_evidence.csv",
    "trip_evidence.csv",
    "contract_rates.csv"
]

for filename in required_files:
    path = DATA_DIR / filename
    if path.exists():
        print(f"Found: {filename}")
    else:
        print(f"Missing: {filename}")

Found: invoices.csv
Missing: shipments_evidence.csv
Found: load_evidence.csv
Found: delivery_evidence.csv
Found: trip_evidence.csv
Found: contract_rates.csv


In [50]:
invoices = pd.read_csv(DATA_DIR / "invoices.csv")
shipments = pd.read_csv(DATA_DIR / "shipment_evidence.csv")
loads = pd.read_csv(DATA_DIR / "load_evidence.csv")
delivery = pd.read_csv(DATA_DIR / "delivery_evidence.csv")
trips = pd.read_csv(DATA_DIR / "trip_evidence.csv")
contracts = pd.read_csv(DATA_DIR / "contract_rates.csv")

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [51]:
datasets={
    "invoices": invoices,
    "shipments":shipments,
    "loads":loads,
    "delivery": delivery,
    "trips": trips,
    "contracts": contracts
}
for name, df in datasets.items():
    print(f"{name:12} : {df.shape[0]:,} rows × {df.shape[1]} columns")

invoices     : 85,410 rows × 21 columns
shipments    : 85,410 rows × 28 columns
loads        : 85,410 rows × 12 columns
delivery     : 85,410 rows × 11 columns
trips        : 85,410 rows × 12 columns
contracts    : 58 rows × 12 columns


In [52]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.dtypes)


INVOICES
invoice_id                         object
shipment_id                        object
load_id                            object
customer_id                        object
route_id                           object
contract_id                        object
invoice_date                       object
actual_distance_miles             float64
base_rate_per_mile                float64
calculated_base_charge            float64
fuel_surcharge_rate               float64
calculated_fuel_surcharge         float64
detention_minutes                   int64
detention_free_minutes              int64
detention_rate_per_hour           float64
calculated_detention_charge       float64
calculated_accessorial_charges    float64
calculated_subtotal               float64
minimum_charge                    float64
expected_total                    float64
billed_amount                     float64
dtype: object

SHIPMENTS
shipment_id                object
load_id                    object
trip_id        

In [53]:
date_columns ={
    "invoices": ["invoice_date"],
    "shipments": ["load_date"],
    "loads": ["load_date"],
    "delivery": ["scheduled_datetime", "actual_datetime"],
    "trips": ["dispatch_date"],
    "contracts": ["valid_from", "valid_to"]
}

for column in date_columns["invoices"]:
    invoices[column]=pd.to_datetime(
        invoices[column],
        errors="coerce"
    )

for column in date_columns["shipments"]:
    shipments[column] = pd.to_datetime(
        shipments[column],
        errors="coerce"
    )

for column in date_columns["loads"]:
    loads[column] = pd.to_datetime(
        loads[column],
        errors="coerce"
    )

for column in date_columns["delivery"]:
    delivery[column] = pd.to_datetime(
        delivery[column],
        errors="coerce"
    )

for column in date_columns["trips"]:
    trips[column] = pd.to_datetime(
        trips[column],
        errors="coerce"
    )

for column in date_columns["contracts"]:
    contracts[column] = pd.to_datetime(
        contracts[column],
        errors="coerce"
    )

print("Date conversion completed.")

Date conversion completed.


In [54]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    
    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing)


INVOICES
No missing values.

SHIPMENTS
No missing values.

LOADS
No missing values.

DELIVERY
No missing values.

TRIPS
driver_id     1714
trailer_id    1680
truck_id      1672
dtype: int64

CONTRACTS
No missing values.


In [55]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    
    print(f"{name:12} → {duplicate_count:,} duplicate rows")

invoices     → 0 duplicate rows
shipments    → 0 duplicate rows
loads        → 0 duplicate rows
delivery     → 0 duplicate rows
trips        → 0 duplicate rows
contracts    → 0 duplicate rows


In [56]:
print("Total invoice rows:", len(invoices))
print("Unique invoice IDs:", invoices["invoice_id"].nunique())

duplicate_invoice_ids = invoices["invoice_id"].duplicated().sum()

print("Duplicate invoice IDs:", duplicate_invoice_ids)

Total invoice rows: 85410
Unique invoice IDs: 85410
Duplicate invoice IDs: 0


In [57]:
key_checks = {
    "shipments": "shipment_id",
    "loads": "load_id",
    "trips": "trip_id",
    "contracts": "contract_id"
}

for name, key in key_checks.items():
    df = datasets[name]
    
    print(
        f"{name:12} | "
        f"rows={len(df):,} | "
        f"unique {key}={df[key].nunique():,}"
    )

shipments    | rows=85,410 | unique shipment_id=85,410
loads        | rows=85,410 | unique load_id=85,410
trips        | rows=85,410 | unique trip_id=85,410
contracts    | rows=58 | unique contract_id=58


In [58]:
print("Delivery events:", len(delivery))

print("\nEvents per load:")
print(
    delivery.groupby("load_id")
    .size()
    .describe()
)

Delivery events: 85410

Events per load:
count    85410.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64


In [59]:
invoice_shipments = invoices["shipment_id"].isin(
    shipments["shipment_id"]
).mean() * 100

print(
    f"Percentage of invoices with matching shipment: "
    f"{invoice_shipments:.2f}%"
)

Percentage of invoices with matching shipment: 100.00%


In [60]:
invoice_loads = invoices["load_id"].isin(
    loads["load_id"]
).mean() * 100

print(
    f"Percentage of invoices with matching load: "
    f"{invoice_loads:.2f}%"
)

Percentage of invoices with matching load: 100.00%


In [61]:
invoice_trip_matches = invoices["shipment_id"].isin(
    shipments["shipment_id"]
)

print(
    f"Percentage of invoices with matching shipment/trip evidence: "
    f"{invoice_trip_matches.mean() * 100:.2f}%"
)

Percentage of invoices with matching shipment/trip evidence: 100.00%


In [62]:
contract_match = invoices["contract_id"].isin(
    contracts["contract_id"]
)

print(
    f"Percentage of invoices with matching contract: "
    f"{contract_match.mean() * 100:.2f}%"
)

Percentage of invoices with matching contract: 100.00%


In [63]:
shipment_cols = [
    "shipment_id",
    "load_id",
    "trip_id",
    "customer_id",
    "route_id",
    "load_date",
    "load_type",
    "weight_lbs",
    "pieces",
    "typical_distance_miles",
    "actual_distance_miles",
    "base_rate_per_mile",
    "fuel_surcharge_rate",
    "detention_minutes",
    "fuel_gallons_used",
    "actual_duration_hours",
    "average_mpg",
    "idle_time_hours"
]

shipment_features = shipments[shipment_cols].copy()

print(shipment_features.shape)

(85410, 18)


In [64]:
load_cols = [
    "load_id",
    "customer_id",
    "route_id",
    "load_date",
    "load_type",
    "weight_lbs",
    "pieces",
    "revenue",
    "fuel_surcharge",
    "accessorial_charges",
    "load_status",
    "booking_type"
]
load_features = loads[load_cols].copy()
print(load_features.shape)

(85410, 12)


In [65]:
trip_cols = [
    "trip_id",
    "load_id",
    "driver_id",
    "truck_id",
    "trailer_id",
    "dispatch_date",
    "actual_distance_miles",
    "actual_duration_hours",
    "fuel_gallons_used",
    "average_mpg",
    "idle_time_hours",
    "trip_status"
]

trip_features = trips[trip_cols].copy()
print(trip_features.shape)


(85410, 12)


In [66]:
delivery_agg = (
    delivery
    .groupby("load_id")
    .agg(
        delivery_event_count=("event_id", "count"),
        delivery_detention_minutes=("detention_minutes", "sum"),
        on_time_rate=("on_time_flag", "mean")
    )
    .reset_index()
)


In [67]:
print("Original delivery rows:", len(delivery))
print("Aggregated load rows:", len(delivery_agg))

print(
    delivery_agg[
        [
            "delivery_event_count",
            "delivery_detention_minutes",
            "on_time_rate"
        ]
    ].describe()
)

Original delivery rows: 85410
Aggregated load rows: 85410
       delivery_event_count  delivery_detention_minutes  on_time_rate
count               85410.0                85410.000000  85410.000000
mean                    1.0                  106.602201      0.446107
std                     0.0                   75.349004      0.497090
min                     1.0                    0.000000      0.000000
25%                     1.0                   37.000000      0.000000
50%                     1.0                  105.000000      0.000000
75%                     1.0                  172.000000      1.000000
max                     1.0                  239.000000      1.000000


In [68]:
contract_duplicate_counts = (
    contracts
    .groupby("contract_id")
    .size()
    .sort_values(ascending=False)
)
contract_duplicate_counts.head(20)

contract_id
CONTRACT00001    1
CONTRACT00002    1
CONTRACT00003    1
CONTRACT00004    1
CONTRACT00005    1
CONTRACT00006    1
CONTRACT00007    1
CONTRACT00008    1
CONTRACT00009    1
CONTRACT00010    1
CONTRACT00011    1
CONTRACT00012    1
CONTRACT00013    1
CONTRACT00014    1
CONTRACT00015    1
CONTRACT00016    1
CONTRACT00017    1
CONTRACT00018    1
CONTRACT00019    1
CONTRACT00020    1
dtype: int64

In [69]:
invoice_base = invoices.copy()

print("Invoice rows:", len(invoice_base))
print("Unique invoices:", invoice_base["invoice_id"].nunique())

Invoice rows: 85410
Unique invoices: 85410


In [70]:
print(
    "Unique shipment IDs:",
    shipment_features["shipment_id"].nunique()
)

print(
    "Rows:",
    len(shipment_features)
)

invoice_base = invoice_base.merge(
    shipment_features,
    on=["shipment_id"],
    how="left",
    suffixes=("", "_shipment"),
    validate="one_to_one"
)

print("After shipment merge:", invoice_base.shape)

Unique shipment IDs: 85410
Rows: 85410
After shipment merge: (85410, 38)


In [71]:
invoice_base = invoice_base.merge(
    load_features,
    on=["load_id"],
    how="left",
    suffixes=("", "_load"),
    validate="one_to_one"
)

print("After load merge:", invoice_base.shape)

print(
    "Unique invoices:",
    invoice_base["invoice_id"].nunique()
)

After load merge: (85410, 49)
Unique invoices: 85410


In [72]:
invoice_base = invoice_base.merge(
    trip_features,
    on=["trip_id"],
    how="left",
    suffixes=("", "_trip"),
    validate="one_to_one"
)

print("After trip merge:", invoice_base.shape)

print(
    "Unique invoices:",
    invoice_base["invoice_id"].nunique()
)

After trip merge: (85410, 60)
Unique invoices: 85410


In [73]:
invoice_base = invoice_base.merge(
    delivery_agg,
    on=["load_id"],
    how="left",
    validate="one_to_one"
)

print("After delivery merge:", invoice_base.shape)

print(
    "Unique invoices:",
    invoice_base["invoice_id"].nunique()
)

After delivery merge: (85410, 63)
Unique invoices: 85410


In [74]:
duplicate_style_columns = [
    col for col in invoice_base.columns
    if "_shipment" in col
    or "_load" in col
    or "_trip" in col
]

duplicate_style_columns

['load_id_shipment',
 'customer_id_shipment',
 'route_id_shipment',
 'actual_distance_miles_shipment',
 'base_rate_per_mile_shipment',
 'fuel_surcharge_rate_shipment',
 'detention_minutes_shipment',
 'customer_id_load',
 'route_id_load',
 'load_date_load',
 'load_type_load',
 'weight_lbs_load',
 'pieces_load',
 'load_id_trip',
 'actual_distance_miles_trip',
 'actual_duration_hours_trip',
 'fuel_gallons_used_trip',
 'average_mpg_trip',
 'idle_time_hours_trip']

In [75]:
missing_after_merge = (
    invoice_base
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_after_merge[missing_after_merge > 0]

driver_id     1714
trailer_id    1680
truck_id      1672
dtype: int64

In [76]:
important_columns = [
    "typical_distance_miles",
    "weight_lbs",
    "pieces",
    "fuel_gallons_used",
    "actual_duration_hours",
    "average_mpg",
    "idle_time_hours",
    "delivery_event_count"
]

for col in important_columns:
    if col in invoice_base.columns:
        match_rate = invoice_base[col].notna().mean() * 100
        print(f"{col:35} {match_rate:6.2f}% available")

typical_distance_miles              100.00% available
weight_lbs                          100.00% available
pieces                              100.00% available
fuel_gallons_used                   100.00% available
actual_duration_hours               100.00% available
average_mpg                         100.00% available
idle_time_hours                     100.00% available
delivery_event_count                100.00% available


In [77]:
numeric_columns = invoice_base.select_dtypes(
    include=np.number
).columns

invoice_base[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
actual_distance_miles,85410.0,1436.770610,806.531806,90.00,718.0000,1301.000,2153.0000,3898.50
base_rate_per_mile,85410.0,2.196849,0.366014,1.52,1.8900,2.190,2.4800,2.79
calculated_base_charge,85410.0,3187.713194,1930.800817,144.90,1463.2400,2990.880,4395.8400,8766.98
fuel_surcharge_rate,85410.0,0.248670,0.061423,0.15,0.1900,0.240,0.3000,0.34
calculated_fuel_surcharge,85410.0,365.119715,227.880760,13.50,167.9100,340.560,514.8000,1154.23
detention_minutes,85410.0,106.602201,75.349004,0.00,37.0000,105.000,172.0000,239.00
detention_free_minutes,85410.0,120.000000,0.000000,120.00,120.0000,120.000,120.0000,120.00
detention_rate_per_hour,85410.0,50.000000,0.000000,50.00,50.0000,50.000,50.0000,50.00
calculated_detention_charge,85410.0,25.166207,35.652522,0.00,0.0000,0.000,49.1700,199.17
calculated_accessorial_charges,85410.0,74.553624,73.512670,0.00,0.0000,75.000,150.0000,400.00


In [78]:
checks = {
    "negative_distance": (
        invoice_base["actual_distance_miles"] < 0
    ).sum(),

    "negative_weight": (
        invoice_base["weight_lbs"] < 0
    ).sum(),

    "negative_fuel": (
        invoice_base["fuel_gallons_used"] < 0
    ).sum(),

    "negative_duration": (
        invoice_base["actual_duration_hours"] < 0
    ).sum(),

    "negative_idle": (
        invoice_base["idle_time_hours"] < 0
    ).sum()
}

pd.Series(checks)

negative_distance    0
negative_weight      0
negative_fuel        0
negative_duration    0
negative_idle        0
dtype: int64

In [79]:
zero_checks = {
    "zero_distance": (
        invoice_base["actual_distance_miles"] == 0
    ).sum(),

    "zero_expected_total": (
        invoice_base["expected_total"] == 0
    ).sum(),

    "zero_billed_amount": (
        invoice_base["billed_amount"] == 0
    ).sum(),

    "zero_fuel": (
        invoice_base["fuel_gallons_used"] == 0
    ).sum(),

    "zero_duration": (
        invoice_base["actual_duration_hours"] == 0
    ).sum()
}

pd.Series(zero_checks)

zero_distance          0
zero_expected_total    0
zero_billed_amount     0
zero_fuel              0
zero_duration          0
dtype: int64

In [80]:
invoice_base[
    [
        "billed_amount",
        "expected_total",
        "calculated_subtotal"
    ]
].describe()

,billed_amount,expected_total,calculated_subtotal
count,85410.000000,85410.000000,85410.000000
mean,3652.955994,3652.955994,3652.552740
std,2132.345520,2132.345520,2132.995469
min,250.000000,250.000000,160.200000
25%,1761.202500,1761.202500,1761.202500
50%,3412.500000,3412.500000,3412.500000
75%,5003.330000,5003.330000,5003.330000
max,9948.180000,9948.180000,9948.180000


In [81]:
invoice_base[
    [
        "actual_distance_miles",
        "typical_distance_miles"
    ]
].describe()

,actual_distance_miles,typical_distance_miles
count,85410.000000,85410.000000
mean,1436.770610,1389.270928
std,806.531806,777.368131
min,90.000000,92.000000
25%,718.000000,697.000000
50%,1301.000000,1271.000000
75%,2153.000000,2108.000000
max,3898.500000,3141.000000


In [82]:
operational_columns = [
    "weight_lbs",
    "pieces",
    "fuel_gallons_used",
    "average_mpg",
    "actual_duration_hours",
    "idle_time_hours",
    "detention_minutes"
]

existing_operational_columns = [
    col for col in operational_columns
    if col in invoice_base.columns
]

invoice_base[existing_operational_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
weight_lbs,85410.0,27477.495305,10095.421857,10000.0,18724.0,27482.0,36190.0,45000.0
pieces,85410.0,14.470530,8.070077,1.0,7.0,14.0,21.0,28.0
fuel_gallons_used,85410.0,221.827422,126.558874,12.0,111.8,204.5,326.4,611.9
average_mpg,85410.0,6.501463,0.578091,5.5,6.0,6.5,7.0,7.5
actual_duration_hours,85410.0,25.014674,14.191944,1.4,12.6,23.0,36.9,67.8
idle_time_hours,85410.0,7.010783,2.882488,2.0,4.5,7.0,9.5,12.0
detention_minutes,85410.0,106.602201,75.349004,0.0,37.0,105.0,172.0,239.0


In [ ]:
assert len(invoice_base) == invoices["invoice_id"].nunique(), (
    "ERROR: Final dataset is not one row per invoice."
)

assert invoice_base["invoice_id"].is_unique, (
    "ERROR: invoice_id is not unique."
)

print("One row per invoice confirmed.")

✓ One row per invoice confirmed.


In [84]:
data_quality_summary = pd.DataFrame({
    "metric": [
        "invoice_rows",
        "unique_invoice_ids",
        "shipment_rows",
        "load_rows",
        "trip_rows",
        "delivery_event_rows",
        "contract_rows",
        "final_ml_rows"
    ],
    "value": [
        len(invoices),
        invoices["invoice_id"].nunique(),
        len(shipments),
        len(loads),
        len(trips),
        len(delivery),
        len(contracts),
        len(invoice_base)
    ]
})

data_quality_summary

,metric,value
0,invoice_rows,85410
1,unique_invoice_ids,85410
2,shipment_rows,85410
3,load_rows,85410
4,trip_rows,85410
5,delivery_event_rows,85410
6,contract_rows,58
7,final_ml_rows,85410


In [85]:
prepared_path = OUTPUT_DIR / "prepared_invoice_data.csv"

invoice_base.to_csv(
    prepared_path,
    index=False
)

print("Rows:", len(invoice_base))
print("Columns:", len(invoice_base.columns))

Rows: 85410
Columns: 63
